# 🏛️ UIT Legal IR - Hybrid Retrieval System (Kaggle Edition)

In [ ]:
import os
# Xem cấu trúc thực tế bên trong Dataset
for root, dirs, files in os.walk("/kaggle/input/uit-legal-ir-data"):
    level = root.replace("/kaggle/input/uit-legal-ir-data", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # Chỉ in 2 cấp đầu
        subindent = " " * 2 * (level + 1)
        for f in files[:10]:
            print(f"{subindent}{f}")
        if len(files) > 10:
            print(f"{subindent}... và {len(files)-10} files khác")


In [ ]:
!pip install -q sentence-transformers rank-bm25 pyvi scikit-learn matplotlib tqdm

In [ ]:
import os
import shutil
# === CẤU HÌNH ĐƯỜNG DẪN KAGGLE ===
KAGGLE_DATA_DIR = "/kaggle/input/datasets/thurdayafternoon/uit-legal-ir-data/uit-legal-ir-data"
KAGGLE_MODEL_DIR = "/kaggle/input/datasets/thurdayafternoon/uit-legal-finetuned-model/fine_tuned_vietnamese_bi_encoder"
WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)
# Copy file corpus resolved (cần ghi được) sang working dir
RESOLVED_SRC = os.path.join(KAGGLE_DATA_DIR, "legal_corpus_resolved.json")
RESOLVED_DST = os.path.join(WORK_DIR, "legal_corpus_resolved.json")
if os.path.exists(RESOLVED_SRC) and not os.path.exists(RESOLVED_DST):
    print("📦 Đang copy legal_corpus_resolved.json sang working dir...")
    shutil.copy2(RESOLVED_SRC, RESOLVED_DST)
# Copy các file cache .pkl (nếu có) sang working dir để tái sử dụng
for pkl_file in [
    "bm25_tokenized_cache_pyvi_v2.pkl",
    "corpus_embeddings_bgem3_resolved.pkl",
    "corpus_embeddings_finetuned_resolved.pkl",
    "corpus_embeddings_e5_resolved.pkl",
    "corpus_embeddings_vietnamese_legal_resolved.pkl",
    "hard_negatives.pkl"
]:
    src = os.path.join(KAGGLE_DATA_DIR, pkl_file)
    dst = os.path.join(WORK_DIR, pkl_file)
    if os.path.exists(src) and not os.path.exists(dst):
        print(f"  📦 Copy cache: {pkl_file}")
        shutil.copy2(src, dst)
# Copy mô hình fine-tuned sang working dir (sentence-transformers cần đọc/ghi)
MODEL_LOCAL = os.path.join(WORK_DIR, "fine_tuned_vietnamese_bi_encoder")
if not os.path.exists(MODEL_LOCAL):
    print("📦 Đang copy mô hình Fine-tuned Bi-Encoder sang working dir...")
    shutil.copytree(KAGGLE_MODEL_DIR, MODEL_LOCAL)
print("✅ Thiết lập đường dẫn hoàn tất!")
print(f"  📂 Data dir: {KAGGLE_DATA_DIR}")
print(f"  🤖 Model dir: {MODEL_LOCAL}")
print(f"  💼 Working dir: {WORK_DIR}")

In [ ]:
import urllib.request
# Danh sách các file Python cần copy từ máy local
# (Nếu bạn đã upload code lên GitHub, dùng git clone thay thế)
source_files = [
    "evaluator.py",
    "bm25_retriever.py",
    "dense_retriever.py",
    "hybrid_retriever.py",
    "graph_resolution.py",
    "train_embedding.py",
    "mine_hard_negatives.py",
    "auto_tuner.py",
    "make_submission.py",
    "visualize_tsne.py",
]
# ============================================
# CÁCH 1: Nếu bạn upload code trực tiếp vào Dataset
# ============================================
# for f in source_files:
#     src = os.path.join(KAGGLE_DATA_DIR, f)
#     dst = os.path.join(WORK_DIR, f)
#     if os.path.exists(src) and not os.path.exists(dst):
#         shutil.copy2(src, dst)
#         print(f"  ✅ Copied: {f}")
#     elif os.path.exists(dst):
#         print(f"  ♻️ Exists: {f}")
#     else:
#         print(f"  ⚠️ Missing: {f} (bạn cần paste code thủ công vào cell bên dưới)")
# ============================================
# CÁCH 2: Clone từ GitHub (nếu đã push lên)
# ============================================
!git clone https://github.com/manh123-chatgpt/implement-pp1.git /kaggle/working/src
import shutil, glob
for f in glob.glob("/kaggle/working/src/*.py"):
    shutil.copy2(f, "/kaggle/working/")
    print(f"✅ Copied: {os.path.basename(f)}")
# Rồi copy từ /kaggle/working/src/ sang /kaggle/working/

In [ ]:
import torch
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🎮 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
from data_loader import load_corpus, load_train_data, load_test_data
corpus = load_corpus()
train_data = load_train_data()
test_data = load_test_data()
print(f"\n📊 Corpus: {len(corpus)} văn bản")
print(f"📊 Train: {len(train_data)} câu hỏi")
print(f"📊 Test: {len(test_data)} câu hỏi")

In [ ]:
from hybrid_retriever import HybridSearcher
from evaluator import compute_metrics
from tqdm import tqdm
import json
import zipfile
# --- Đánh giá trên validation set ---
hybrid = HybridSearcher(corpus)
val_items = list(train_data.items())[:500]
val_questions = {k: v["question"] for k, v in val_items}
val_truth = {k: v["answer"] for k, v in val_items}
print("\n🔍 Đang đánh giá trên 500 câu hỏi validation...")
val_preds = {}
for qid, question in tqdm(val_questions.items(), desc="Hybrid Validation"):
    val_preds[qid] = hybrid.search(question, top_k=5)
results = compute_metrics(val_preds, val_truth, k=5)
print(f"\n📊 KẾT QUẢ HYBRID SEARCH:")
print(f"   Recall@5   : {results['Recall'] * 100:.2f}%")
print(f"   Precision@5: {results['Precision'] * 100:.2f}%")
# --- Tạo Submission cho Public Test ---
print("\n📦 Đang sinh kết quả cho Public Test...")
submission = {}
for qid, item in tqdm(test_data.items(), desc="Predicting"):
    question = item["question"]
    top_docs = hybrid.search(question, top_k=5)
    submission[str(qid)] = {"answer": [str(d) for d in top_docs[:5]]}
with open("submission.json", "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)
with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("submission.json", arcname="submission.json")
print(f"\n🎉 Đã tạo submission.zip! Download từ tab Output của Kaggle.")

In [ ]:
from visualize_tsne import run_tsne_visualization
run_tsne_visualization(
    model_path="fine_tuned_vietnamese_bi_encoder",
    num_samples=200,
    output_image="embedding_tsne_visualization.png"
)
from IPython.display import Image, display
display(Image("embedding_tsne_visualization.png", width=900))

In [ ]:
# Bước 1: Khai thác Hard Negatives đa tầng
from mine_hard_negatives import mine_multi_stage_hard_negatives
mine_multi_stage_hard_negatives()

In [ ]:
# Bước 2: Huấn luyện lại mô hình Bi-Encoder với Hard Negatives mới
# ⚠️ Cần ~4-5 tiếng GPU T4, chạy qua đêm nếu cần
from train_embedding import train
train()